# CellCommunication

Modular ChronoPack cell-cell communication inference for the THP1 -> K562 setting.

This notebook keeps the core workflow: load CellChat secreted ligand-receptor pairs, filter to expressed sender/receiver components, restrict to temporally associated ligands, score early ligand-receptor activity, and generate ChronoPack ligand rankings. Exploratory one-off plotting from the original notebook has been reduced to optional summary plots.

## Setup

In [ ]:
from pathlib import Path
import os
import sys

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

_start = Path.cwd().resolve()
_repo_root = next((p for p in [_start, *_start.parents] if (p / "chronopack").exists()), _start)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from chronopack.c2c import (
    build_pair_series,
    filter_expressed_secreted_lr_catalog,
    load_secreted_cellchat_pairs,
    load_tradeseq_grid_exports,
    plot_avg_vs_logp,
    plot_genes_over_time,
    plot_pair_overlay,
    restrict_ligands_to_association,
    score_lr_pairs,
    split_by_cell_type,
)

sc.settings.verbosity = 3
sc.set_figure_params(figsize=(5, 5))

## Configuration

In [ ]:
WORKSPACE_DIR = Path.cwd().resolve()
NOTEBOOK_DIR = WORKSPACE_DIR if WORKSPACE_DIR.name == "c2c" else WORKSPACE_DIR / "c2c"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_NOTEBOOK = Path("/new-stg/home/alex/tmp/c2c_downsample/CellCommunication.ipynb")
SOURCE_DIR = SOURCE_NOTEBOOK.parent

def original_project_dir() -> Path:
    for marker_name in ("original_dir.txt", "OriginalDirectoryPath.txt"):
        marker = NOTEBOOK_DIR / marker_name
        if marker.exists():
            raw = Path(os.path.expanduser(marker.read_text().strip()))
            return raw.parent.parent if raw.suffix == ".ipynb" or raw.is_file() else raw
    return Path("/new-stg/home/alex/projects/chrono_seq/analyses/thp1-k562-lps.hg38.rep")

PROJECT_DIR = original_project_dir()
REPRO_INPUT_DIR = NOTEBOOK_DIR / "reproduction" / "input_files"

def input_file(name: str, fallback: Path) -> Path:
    local = REPRO_INPUT_DIR / name
    return local if local.exists() else fallback

CELLCHAT_PATH = input_file(
    "interaction_input_CellChatDB.csv",
    Path("/new-stg/home/alex/ref/cellchatdb/hs/interaction_input_CellChatDB.csv"),
)
ADATA_PATH = input_file(
    "adata_full.chk1.h5ad",
    PROJECT_DIR / "output" / "adata_full.chk1.h5ad",
)
ASSO_BOTH_PATH = input_file(
    "asso_df.both.2trajectory.nknots4.expanded.csv",
    PROJECT_DIR / "intmd" / "sc_to_tradeSeq" / "asso_df.both.2trajectory.nknots4.expanded.csv",
)
TS_EXPORT_GRID_DIR = PROJECT_DIR / "intmd" / "sc_to_tradeSeq" / "ts_export_grid_100pts"

SENDER_CELL_TYPE = "THP1"
RECEIVER_CELL_TYPE = "K562"
CELL_TYPE_KEY = "cell_type"
COUNTS_LAYER = "counts"
EXPR_LAYER = "log1p"
MIN_DETECT = 1e-100
CAUSAL_WINDOW_MIN = (0, 45)
DOWNSAMPLE_INTERVAL_MIN = 15
PREFER_FITS_FOR_SCORING = False
RESPONSE_GENES = ["NFKBIA", "NFKBIZ", "TNFAIP3", "BIRC3", "IER3", "CXCL1", "CXCL2", "CXCL3"]

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"ADATA_PATH: {ADATA_PATH}")
print(f"ASSO_BOTH_PATH: {ASSO_BOTH_PATH}")
print(f"TS_EXPORT_GRID_DIR: {TS_EXPORT_GRID_DIR}")

## Load Inputs

In [ ]:
cellchat_pairs = load_secreted_cellchat_pairs(CELLCHAT_PATH)
adata_full = sc.read_h5ad(ADATA_PATH)
adata_thp1, adata_k562 = split_by_cell_type(
    adata_full,
    cell_type_key=CELL_TYPE_KEY,
    sender=SENDER_CELL_TYPE,
    receiver=RECEIVER_CELL_TYPE,
)
tradeseq_exports = load_tradeseq_grid_exports(TS_EXPORT_GRID_DIR)
fits = tradeseq_exports["fits"]
wmeans = tradeseq_exports["wmeans"]
asso_df = pd.read_csv(ASSO_BOTH_PATH, index_col=0)

print(f"Secreted CellChat pairs: {cellchat_pairs.shape}")
print(f"AnnData full: {adata_full.shape}; THP1: {adata_thp1.shape}; K562: {adata_k562.shape}")
print(f"tradeSeq fits: {fits.shape}")
print(f"Association table: {asso_df.shape}")

## Expressed Secreted Ligand-Receptor Catalog

In [ ]:
lr_expressed, det_thp1, det_k562 = filter_expressed_secreted_lr_catalog(
    cellchat_pairs,
    sender_adata=adata_thp1,
    receiver_adata=adata_k562,
    sender_layer=COUNTS_LAYER,
    receiver_layer=COUNTS_LAYER,
    min_detect=MIN_DETECT,
)
lr_scored_catalog = restrict_ligands_to_association(lr_expressed, asso_df)

print(
    f"Kept {len(lr_expressed)} of {len(cellchat_pairs)} secreted interactions with all components expressed "
    f"(ligand in {SENDER_CELL_TYPE}, receptor in {RECEIVER_CELL_TYPE}; min_detect={MIN_DETECT:.2g})."
)
print(f"Kept {len(lr_scored_catalog)} interactions after requiring ligand temporal association rows.")
lr_scored_catalog[["ligand", "receptor", "pathway_name"]].head(10)

## ChronoPack Ligand-Receptor Scoring

In [ ]:
pair_series = build_pair_series(
    lr_scored_catalog,
    sender_adata=adata_thp1,
    receiver_adata=adata_k562,
    response_genes=RESPONSE_GENES,
    time_key="time",
    layer=EXPR_LAYER,
    prefer_fits=PREFER_FITS_FOR_SCORING,
    fits_df=fits,
    wmeans_df=wmeans,
)

plot_df, chronopack_ligand_scores = score_lr_pairs(
    pair_series,
    association_df=asso_df,
    receiver_detection=det_k562,
    causal_window_min=CAUSAL_WINDOW_MIN,
    association_p_col="pvalue_2",
)

pair_scores_out = plot_df.drop(columns=["pair_rank"]).copy()
pair_scores_out.insert(0, "interval_min", DOWNSAMPLE_INTERVAL_MIN)
pair_scores_out["logp"] = pair_scores_out["logp"].round(12)

pair_scores_out.to_csv(OUTPUT_DIR / "chronopack_lr_pair_scores.csv", index=False)
plot_df.to_csv(OUTPUT_DIR / "chronopack_lr_pair_scores.with_rank.csv", index=False)
chronopack_ligand_scores.to_csv(OUTPUT_DIR / "chronopack_ligand_scores.csv", index=False)

print(f"Scored LR pairs: {plot_df.shape}")
print(f"Scored ligands: {chronopack_ligand_scores.shape}")
chronopack_ligand_scores.head(20)

## Candidate Pair Summary Plot

In [ ]:
plot_df["rec_det_clipped"] = np.clip(plot_df["rec_det"], 0, 0.2)
fig, ax = plot_avg_vs_logp(
    plot_df,
    color_col="rec_det_clipped",
    p_thresh=0.05,
    x_thresh=0.1,
    max_labels=20,
    title="ChronoPack LR pair prioritization",
    xlabel="early LR activity (0-45 min)",
    ylabel="ligand -log10(p)",
)
fig.savefig(OUTPUT_DIR / "chronopack_lr_pair_scores.png", dpi=300, bbox_inches="tight")

## Optional TNF/NF-kB Overlay

In [ ]:
tnf_pairs = pd.DataFrame({"ligand": ["TNF", "TNF"], "receptor": ["TNFRSF1A", "TNFRSF1B"]})
tnf_pair_series = build_pair_series(
    tnf_pairs,
    sender_adata=adata_thp1,
    receiver_adata=adata_k562,
    response_genes=["NFKBIA"],
    time_key="time",
    layer=EXPR_LAYER,
    prefer_fits=True,
    fits_df=fits,
    wmeans_df=wmeans,
)

if ("TNF", "TNFRSF1A") in tnf_pair_series:
    fig, ax = plot_pair_overlay(
        tnf_pair_series,
        pair=("TNF", "TNFRSF1A"),
        norm_mode="minmax",
        nfkb_onset_time=70,
        figsize=(4, 2.5),
        title="TNF->TNFRSF1A and K562 NFKBIA",
    )
    fig.savefig(OUTPUT_DIR / "TNF_TNFRSF1A_NFKBIA_overlay.png", dpi=300, bbox_inches="tight")
else:
    print("TNF->TNFRSF1A was not available for overlay plotting.")

## Optional Expression Sanity Plots

In [ ]:
genes = ["TNF", "TNFRSF1A", "TNFRSF1B", "NFKBIA", "TNFAIP3", "BIRC3"]
genes = [gene for gene in genes if gene in adata_full.var_names]
fig, axes = plot_genes_over_time(
    adata_full,
    genes,
    time_key="time",
    cluster_key=CELL_TYPE_KEY,
    layer=EXPR_LAYER,
    ncols=3,
)
fig.savefig(OUTPUT_DIR / "selected_gene_expression_over_time.png", dpi=300, bbox_inches="tight")